In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic"
os.environ['MKL_THREADING_LAYER'] = "GNU"

In [3]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
from concept_abstraction.environments import ConceptEnv
import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [4]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [5]:
if is_main:
    if is_jupyter: 
        # Basics 
        seed        = 42
        environment_string = "cart_pole"
        gold_timesteps =4_000_000
        training_timesteps = 250_000
        num_concepts_selected = 3
        selection_function = "q_value"
        # Experiment #1 & #2
        run_basic = False
        run_iterative = False 
        run_two_stage = True   
        run_imperfect=False 
        run_intervention=False 
        # Experiment #3
        cbm_accuracy_by_concept = None 
        intervention_probability = 0
        intervention_accuracy_by_concept = None 
        cbm_std_by_concept = None 
        target_abstraction = 0.05
        reward_error = 0
        # Experiment #4
        concept_source = "human_selected_binary"
        # Experiment #5
        assess_completeness=False
        # Experiment #6
        num_iterations = 0
        selections_per_round = 0
        initial_concepts = 0
        out_folder = "llm"
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument('--seed', help='Random Seed', type=int, default=42)
        parser.add_argument('--environment_string', help='Which environment to create', type=str, default="tree")
        parser.add_argument('--training_timesteps', help='Number of training timesteps', type=int, default=10000)
        parser.add_argument('--gold_timesteps', help='Number of training timesteps without concepts', type=int, default=10000)
        parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
        parser.add_argument('--selection_function', help='When selecting, use q_value, policy, or transition?', type=str, default="policy")
        parser.add_argument('--cbm_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--intervention_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--cbm_std_by_concept', help="What is the error of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--run_two_stage', help='Run the two stage?', action='store_true')
        parser.add_argument('--run_iterative', help='Run the iterative?', action='store_true')
        parser.add_argument('--run_intervention', help='Run the intervention?', action='store_true')
        parser.add_argument('--run_basic', help='Run the basic comparisons?', action='store_true')
        parser.add_argument('--run_imperfect', help='Run the imperfect comparisons?', action='store_true')
        parser.add_argument('--intervention_probability', help='Value for the target abstraction with human performance', type=float, default=0.05)
        parser.add_argument('--target_abstraction', help='Value for the target abstraction with human performance', type=float, default=0.05)
        parser.add_argument('--reward_error', help="How much to perturb the reward by?", type=float, default=0)
        parser.add_argument('--concept_source', help='When selecting, use q_value, policy, or transition?', type=str, default="human_selected")
        parser.add_argument('--assess_completeness', help='Compare to the concept completeness algorithm?', action='store_true')
        parser.add_argument('--num_iterations', help='Number of iterations for iterative algorithms',type=int, default=0)
        parser.add_argument('--selections_per_round', help='Concepts to select per round',type=int, default=0)
        parser.add_argument('--initial_concepts', help='Number of starting/initial concepts',type=int, default=0)
        parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

        args = parser.parse_args()

        seed = args.seed
        environment_string = args.environment_string
        training_timesteps = args.training_timesteps 
        gold_timesteps = args.gold_timesteps
        num_concepts_selected = args.num_concepts_selected
        selection_function = args.selection_function
        cbm_accuracy_by_concept = args.cbm_accuracy_by_concept
        cbm_std_by_concept = args.cbm_std_by_concept
        run_basic = args.run_basic
        run_iterative = args.run_iterative
        run_two_stage = args.run_two_stage
        run_imperfect = args.run_imperfect
        run_intervention = args.run_intervention
        intervention_probability = args.intervention_probability
        intervention_accuracy_by_concept = args.intervention_accuracy_by_concept
        target_abstraction = args.target_abstraction
        reward_error = args.reward_error
        concept_source = args.concept_source
        assess_completeness = args.assess_completeness
        num_iterations = args.num_iterations 
        selections_per_round = args.selections_per_round
        initial_concepts = args.initial_concepts
        out_folder = args.out_folder

    save_name = secrets.token_hex(4)  

In [6]:
if is_main:
        results = {}
        results['parameters'] = {'seed'      : seed,
                'environment_string'    : environment_string, 
                'training_timesteps': training_timesteps, 
                'gold_timesteps': gold_timesteps,
                'selection_function': selection_function,
                'num_concepts_selected': num_concepts_selected,
                'cbm_accuracy_by_concept': cbm_accuracy_by_concept,
                'cbm_std_by_concept': cbm_std_by_concept,
                'intervention_probability': intervention_probability,
                'intervention_accuracy_by_concept': intervention_accuracy_by_concept,
                'target_abstraction': target_abstraction,
                'reward_error': reward_error, 
                'concept_source': concept_source,
                'assess_completeness': assess_completeness,
                'num_iterations': num_iterations,
                'selections_per_round': selections_per_round, 
                'initial_concepts': initial_concepts,
                'run_basic': run_basic,
                'run_iterative': run_iterative, 
                'run_two_stage': run_two_stage, 
                'run_intervention': run_intervention,
                'run_imperfect': run_imperfect, 
        }
        print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'environment_string': 'cart_pole', 'training_timesteps': 250000, 'gold_timesteps': 4000000, 'selection_function': 'q_value', 'num_concepts_selected': 3, 'cbm_accuracy_by_concept': None, 'cbm_std_by_concept': None, 'intervention_probability': 0, 'intervention_accuracy_by_concept': None, 'target_abstraction': 0.05, 'reward_error': 0, 'concept_source': 'human_selected_binary', 'assess_completeness': False, 'num_iterations': 0, 'selections_per_round': 0, 'initial_concepts': 0, 'run_basic': False, 'run_iterative': False, 'run_two_stage': True, 'run_intervention': False, 'run_imperfect': False}


In [7]:
if is_main:
    np.random.seed(seed)
    random.seed(seed)

In [8]:

if is_main:
    np.random.seed(seed)
    random.seed(seed)

if is_main:
    concept_list = get_concepts(environment_string,"human_selected_binary",seed)
    num_concepts_selected = min(num_concepts_selected,len(concept_list))
    ground_truth_env, ground_truth_gym_env, additional_info = get_environment(environment_string, None, seed)   
    model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,gold_timesteps,seed)
    if os.path.exists(model_name):
        groundtruth_model = PPO.load(model_name)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [9]:

if is_main:
    model_name = "../../results/models/concept_predictor_env={}_training={}_seed={}.pth".format(environment_string,100,seed)

    height = width = 84

    if environment_string == "mini_grid":
        num_frames = 1
    else:
        num_frames = 4

    if environment_string == "cart_pole":
        height = 160
        width = 240

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if os.path.exists(model_name):
        concept_predictor = ConceptPredictorCNN(len(concept_list), num_frames=num_frames,height=height,width=width).to(device)
        concept_predictor.load_state_dict(torch.load(model_name, weights_only=True))
        concept_predictor.eval()
    else:
        concept_predictor, acc_list = train_concept_predictor(ground_truth_gym_env,groundtruth_model,concept_list,list(range(len(concept_list))),environment_string,epochs=25,max_episode_length=10_000)
        torch.save(concept_predictor.state_dict(), model_name)
        concept_predictor.eval()

In [10]:

if is_main:    
    model_name = "../../results/q_estimates/env={}_training={}_seed={}_selection={}_source={}.pkl".format(environment_string,gold_timesteps,seed,"q_value","human_selected_binary")
    if os.path.exists(model_name):
        q_estimates = pickle.load(open(model_name,"rb"))
    else:
        q_estimates = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list)
        pickle.dump(q_estimates,open(model_name,"wb"))

In [11]:
acc_list = [0.9765814019455707, 0.9809948106989704, 0.8643189668118336, 0.8645358829320385, 0.8642855951010329, 0.867522651048706, 0.9678964142096744, 0.9561829437186097, 0.8658540655086683, 0.8587959486743088, 0.859513440456525, 0.8634846740418147]

In [12]:
subset_concept, idx = concept_completeness_selection(ground_truth_env,concept_list,num_concepts_selected,"q_value",q_estimates,"human_selected_binary")
subset_concept

[<function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>,
 <function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>,
 <function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>]

In [13]:
two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string,subset_concept,seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=idx)
two_stage_env.reset()

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

array([[-0.6974249 ,  0.53867567,  0.6018259 ],
       [-0.15058571,  0.63315964,  0.35630926],
       [ 0.5822468 , -0.67504555,  1.0703213 ],
       [-0.42481124,  0.39364278, -1.3859612 ],
       [-0.43897206, -0.02626934,  0.3990263 ],
       [-0.3163289 ,  0.49410257, -1.5321392 ],
       [-0.9550849 ,  0.98375374, -0.7079459 ],
       [ 2.4011827 , -2.3419285 ,  1.1985787 ]], dtype=float32)

In [93]:
two_stage_env.reset()

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated



array([[ 0.54705375,  0.867506  , -0.06649437],
       [ 0.6994276 ,  1.0496479 , -2.1531036 ],
       [ 0.54705375,  0.867506  , -0.06649437],
       [-2.1467664 ,  0.12449791,  0.51667327],
       [ 0.43310165, -1.6449741 , -0.12133817],
       [-1.2112397 , -1.6150056 ,  1.3974483 ],
       [ 0.49887165,  0.14450206,  0.9531159 ],
       [ 0.6325646 ,  0.20637636, -0.45985788]], dtype=float32)

In [51]:
two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string,subset_concept,seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=idx,intervention_prob=0)


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [40]:
intervention_prob = 0.0

In [41]:
two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string,subset_concept,seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=idx,intervention_prob=intevrvention_prob)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=250_000,custom_name="cart_pole_test")   
performance = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed,max_steps=25_000)
performance

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated



approx_kl,█▅▅▅▂▃▂▂▃▄▃▃▂▂▁▃▂▂▂▁▂▂▂▂▂▃▂▂▂▄▂▄▃▃▂▂▃▃▂▂
clip_fraction,▇▇▆▅▄▁▃▄▂▂▄▄▃▄▄▂▃▂█▂▃▃▂▂▃▂▂▄▄▃▃▂▂▂▂▁▂▃▂▂
ema_norm_reward,▁▁▁▁▁▂▂▂▃▄▆▇▆▇▇█▆▆▅▆█▇▆▇█▆▆▆▆▆█████▇███▅
entropy_loss,▁▂▅▅▆▇▇▇▇█▇▇██████▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇███
episode_length_mean,▁▁▁▁▁▁▂▂▁▃▃▄▄▅▄▅█▆▇▄▆▇████▄▃▅▅█▃█▆▅▂▇▅▄▅
episode_reward_max,▁▁▂▁▁▁▁▂▁▃▃▆▄▄▅▅█▇▄▃▅██▃█▄█████▆█▆███▄▆▄
episode_reward_mean,▁▁▂▁▁▁▁▁▂▂▂▂▂▁▃▄▃▄▃▄▇▇█▃████▆▄█▇█▇▆█▇██▆
episode_reward_min,▁▁▁▁▁▂▂▂▃▂▂▂▃▆▄▅▇█▄▃▇█▃▆▇▆█▇██▇▆▆██▄███▂
episodes_completed,▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▅▆▆▇▇▇▇▇▇▇██
explained_variance,▂▃▃▂▂▂▁▁▃▃▄▅▅▅▅▂▂▃▄▄▃▃▂▂▂▃▃▃▄▄▅▆▇██▇▇▇▇▆
+1,...


299.4935064935065

In [42]:
intevrvention_prob = 0.5

In [43]:
two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string,subset_concept,seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=idx,intervention_prob=intevrvention_prob)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=250_000,custom_name="cart_pole_test")   
performance = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed,max_steps=25_000)
performance

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated



approx_kl,█▄▃▃▃▃▂▂▁▄▄▃▃▅▂▂▂▂▃▃▃▃▃▃▃▂▂▃▃▃▂▂▄▃▃▃▃▃▁▄
clip_fraction,██▄▄▃▁▁▁▁▂▁▂▂▂▃▂▂▁▁▂▂▁▁▂▃▂▂▂▁▁▁▂▂▂▂▃▂▂▂▃
ema_norm_reward,▁▁▁▁▁▁▁▁▁▁▂▂▂▃▃▄▄▆▇▇▇▇▆▆▇▇▇▆▆▆▆▆▆▆▆▇▇▇██
entropy_loss,▁▃▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇██▇▇▇▇▇▇▇▇▇▇▇▇████████
episode_length_mean,▁▁▁▂▁▂▂▂▂▄▄▇▇▄█▅▆▅▇██████▆▄█▄▂█▄██▄▅██▇▇
episode_reward_max,▁▂▁▁▁▁▁▁▁▁▁▇▂▃▃█▃██▇▃▇██▆█▆██▇▄█▇██▄▅███
episode_reward_mean,▁▁▁▁▁▁▁▂▁▂▂▂▂▃▆▅▅▆██▆▇███████▆█▄▅▄▃▅▆█▃█
episode_reward_min,▁▁▁▁▁▁▁▁▃▁▄▃▃▂▂▃▆███▇▄███████▂▆▄▅█████▃█
episodes_completed,▁▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇████
explained_variance,▂▂▂▃▁▁▁▁▄▅▃▃▃▃▂▃▄▄▄▆▅▇██▆▆▆▆▃▃▄▄▅▅█▅▃▃▃▃
+1,...


439.5

In [31]:
env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
model = train_ppo_model(env,environment_string,policy="MlpPolicy",total_timesteps=250_000,custom_name="cart_pole_test")  
performance = evaluate_model(environment_string,eval_env,additional_info,model,seed,max_steps=25_000)
performance 

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

approx_kl,██▃▃▅▄▄▂▄▄▁▁▁▂▅▁▄▃▂▂▅▃▃▄▁▄▄▄▄▄▃▃▃▃▃▆▆▆▃▃
clip_fraction,▃▆▂▂▅▄▂▂▅▃▅▅▅▅▃▂▁▁▁▃▃███▄▁▁▁▁▄▂▂▂▄▁▁▇▆▆▄
ema_norm_reward,▁▁▁▁▁▂▁▂▂▂▃▃▃▃▃▄▅▅▅▆▇▆▇▅▆▇▇▇▆▇▇▇▆▆▇▆▆▇█▇
entropy_loss,▁▃▃▄▄▄▄▄▅▅▆▆▇▇▇▇█▇▇▇▆▆▆▆▆▅▅▅▅▅▅▆▆▆▆▆▆▅▅▅
episode_length_mean,▁▁▁▁▁▂▂▁▁▃▃▂▃▃▆▂▅▄▄▃▆▇▇▄▆█▇▆▃▇▆▄▇▆▆▃▇▃▇▅
episode_reward_max,▁▂▁▂▁▂▁▄▇▃▅█▄▄█▆▅▄█▇█▅▄▅▄▄▆▃▄▇▄▄▄▅▄▄██▇▄
episode_reward_mean,▁▁▁▁▁▁▂▂▁▁▂▂▃▃▃▄▄▄▄▃▃▃█▄▆▄▄▆█▆▅▅▄▄▄██▅▅▄
episode_reward_min,▁▂▁▁▁▁▁▁▄▄▃▄▃▄▁▅▃▆▂█▄▇█▅▇▅▃▆▄▄█▃▄▂▄▅▇█▁▅
episodes_completed,▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█
explained_variance,▅▅▅▁▆▅▅▆▅▆▆▆▅▅▅▅▆▅▅▅▇▇▅▅▅▆▆▅▅▅▅▅▅▅▆▇█▄▄▄
+1,...


245.94791666666666

approx_kl,▇▄▅▄▄█▄▄▂▅▆▆▅▅▅▃▇▆▄▄▃▄▂▂▂▃▁▃▃▄▃▃▄▂▄▄▃▅▄▂
clip_fraction,▇▇▇▆▆▇▇▂▂▂▂▂██▅▆▇▃▃▃▃▄▄▂▄▁▂▅▅▂▅▁▂▂▄▄▄▃▃▃
ema_norm_reward,▁▁▁▁▁▁▁▁▂▂▂▂▄▃▄▅▅▅▆▆▆█▆▆▆▆▆▆▆▇█▇█▇▇▇▇█▆█
entropy_loss,▁▁▂▂▂▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇█▇██▇████▇█▇▇▇
episode_length_mean,▁▁▁▁▁▁▂▁▁▂▁▂▁▂▄▂▄▅▃▂▅█▄▄▄▆▄▄▄▄▅▅▄▆▄▄▆▅▇▅
episode_reward_max,▁▂▁▁▁▁▂▁▁▃▁▁▃▂▄▅▄▆▅▄▇▂▄▃▅█▆▅▅▃▄▆▄▆▄▇▇▄▅▄
episode_reward_mean,▁▁▁▁▁▁▁▁▁▁▁▂▂▂▄▇▅▄▄▄▂▃▃▄▆█▅▇█▆▅▆▅▇▅▄▅▇▆▄
episode_reward_min,▁▁▁▁▁▁▁▁▁▁▃▁▂▄▆▆▄▆▄▄▅█▄▃▆▄▅▃▆▆▅▅▇▆█▄▅▆▄▄
episodes_completed,▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇████
explained_variance,▅██▆▇▁▂▃▁▁▁▁▂▃▃▃▅▆▇▇▅▇▆▆▅▃▄▄▄▅▃▃▃▃▄▄▄▅▄▇
+1,...


In [41]:
intervention_prob = 0.5
intervene_concepts = [int(np.random.random() < 0.5) for i in range(len(concept_list))]
accuracies = [1 if intervene_concepts[i] else acc_list[i] for i in range(len(acc_list))]
subset_concept, idx = multiple_lp_selection(ground_truth_env,concept_list,num_concepts_selected,"q_value",q_estimates,"human_selected_binary",accuracies)


Final vals 10076
Optimizing


In [47]:
two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string,subset_concept,seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=idx,intervention_prob=intervention_prob)


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [50]:
two_stage_env.intervene_concepts = [intervene_concepts[i] for i in idx]

In [51]:
two_stage_gym_env.intervene_concepts = [intervene_concepts[i] for i in idx]

In [52]:

model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_multiple".format(environment_string))    
multiple_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)


approx_kl,▂▂▂▂▂▃▃▃▃▃▃▃▂▃▃▂▁▁▁▂▂▂▃▂▃▃▃▅▅▅▅▄▅▅▅▇▇█▆▆
clip_fraction,▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▂▁▁▁▁▁▁▁▁▁▃▃▂▂▂▂▂▅▅▅▅▅▅█▅
ema_norm_reward,▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▄▄▄▅▅▅▇▅▆██▇▇▇▇▇█████
entropy_loss,▁▁▂▂▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▄▅▅▇▇█
episode_length_mean,▇██████████████▅▆▄█▄█▆▃▃▂▄█▃▁▁▁▃▁▃▁▃▁▅▂▁
episode_reward_max,▆▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▆▁▁▃▄▁▇▇█▆▅▅▅▆▇███▇██
episode_reward_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▇▁▁▁▁▁▁▁▁▁▁▇▇▄▆▁█▇▄▆█▇▆
episode_reward_min,▇▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▅▁▆▁▆█▆▄█▃▇▄▇▅
episodes_completed,▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇█
explained_variance,▁▅▅▇▇▆▇▅▅▅▄▇▇▇▇▆▆▆▅▆▄████▇▇▇████████████
+1,...


KeyboardInterrupt: 

In [33]:
wandb.finish()

approx_kl,▄▄▄▂▂▁▁▁▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄████▁▁▁▇▇▇▇▁▁▁▅▅▅
clip_fraction,▂▂▂▂▂▂▂▁▁▁▅▅▅▅▅▅▂▂▂▂███▁▁▁▁▁▁▄▄▄▄▄▄▁▁▁▁▅
ema_norm_reward,▄▃▂█▇▂▂▇▂▄▂▂▂▁▁▂▁▁▁▅▃▇▅▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
entropy_loss,▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████
episode_length_mean,▁███████████████████████████████████████
episode_reward_max,▄▁▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
episode_reward_mean,▆▁▁▁▁▁▁▁▁▁▁▁▁▆▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
episode_reward_min,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
episodes_completed,▁▁▁▁▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇██
explained_variance,▁▁▁▁▁▅▅▅▅▇▇▇▇▇▇███████▅▅▅▅▅▅▄▄▄▄▄▄▄▄▇▇▇▇
+1,...


In [ ]:
multiple_two_stage_reward

In [34]:
two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string,subset_concept,seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=idx,intervention_prob=0)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_multiple".format(environment_string))    
multiple_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

KeyboardInterrupt: 

In [ ]:
multiple_two_stage_reward

In [ ]:
subset_concept, idx = lp_based_selection(ground_truth_env,concept_list,num_concepts_selected,"q_value",q_estimates,"human_selected_binary")
two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string,subset_concept,seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=idx,intervention_prob=0.75)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_lp".format(environment_string))    
lp_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)
lp_two_stage_reward

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

approx_kl,▃▃▂▂▂▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▃▂▄▄▃▃▃▅▅▆▇▆▆▆▇██▆▆█
clip_fraction,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▁▁▃▃▃▃▄▅▆▆▆▆███▇▇▅▅▅▅▆
ema_norm_reward,▁▁▁▁▂▁▁▁▁▁▃▁▃▃▂▅▄▄▆▆▆▆▇▇▇▇▇▇█▇██████████
entropy_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▅▅▅▅▆▇▇▇▇█
episode_length_mean,▇█████████▆▇█▃█▃▄▇▃▃▃▂▂▁▁▂▂▃▂▁▁▂▁▁▁▁▁▂▁▂
episode_reward_max,▁▁▄▁▁▁▅▁▆▆█▃▇▆▇▅█████▆██▅███████████████
episode_reward_mean,▂▁▁▁▁▁▁▃▂▂▃▇▅▅▇▇▇▆████▇▇▇██▇██████▇███▇█
episode_reward_min,▁▁▁▁▁▁▁▁▇▁▅▆▃▇▇▆▆▅█▆▅███▇▆▇█▆███▇▇█▇██▇█
episodes_completed,▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇███
explained_variance,▁▃▃▅▅▃▃▃▃▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇█▇██████████
+1,...


We had [0.964, 0.9604, 0.9568, 0.9532, 0.9496, 0.9316, 0.9172, 0.9568, 0.9568, 0.9604, 0.9676, 0.946, 0.9604, 0.9028, 0.9424, 0.9568, 0.946, 0.9424, 0.9532, 0.8164, 0.8956, 0.9532, 0.964, 0.9424, 0.964, 0.9388, 0.9424, 0.9388, 0.9496, 0.9676, 0.9208000000000001, 0.9676, 0.964, 0.964, 0.9244, 0.946, 0.9424, 0.8992, 0.9496, 0.9316, 0.6832, 0.964, 0.9496, 0.9532, 0.9424, 0.946, 0.9676, 0.9064, 0.9388, 0.946, 0.9352, 0.9496, 0.91, 0.9244, 0.9279999999999999, 0.9028, 0.784, 0.8992, 0.9496, 0.9316, 0.9316, 0.8452, 0.91, 0.9496, 0.9424, 0.8956, 0.9352, 0.9244, 0.946, 0.9388, 0.9532, 0.9388, 0.9568, 0.9316, 0.9424, 0.9604, 0.946, 0.9279999999999999, 0.9244, 0.8884, 0.9604, 0.9604, 0.964, 0.946, 0.7876, 0.9496, 0.9064, 0.946, 0.9604, 0.9244, 0.9028, 0.9568, 0.9496, 0.9496, 0.9712, 0.8596, 0.964, 0.9604, 0.9388, 0.946, 0.964, 0.9244, 0.9496, 0.9676, 0.9604, 0.9604, 0.9244, 0.9532, 0.946, 0.7336, 0.9496, 0.9676, 0.9316, 0.9604, 0.8884, 0.91, 0.9136, 0.9279999999999999, 0.9496, 0.946, 0.9676, 0.93

0.9319424673109721

In [ ]:
a,b = two_stage_gym_env.reset()
a,b,c,d,e = two_stage_gym_env.step([2 for i in range(8)])

In [ ]:
a_pred = np.array([[concept_list[j](b[i]['observation']) for j in idx] for i in range(len(b))])
np.mean(a == a_pred)

KeyError: 'observation'

In [ ]:
model = train_ppo_model(env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_lp".format(environment_string))    
lp_two_stage_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
lp_two_stage_reward

approx_kl,▁▁▁▁▂▃▃▂▁▂▂▁▁▃▄▇▇▁▁▁▂▃▃▃█▄▂▂▂▃▃▅▅▆▆▆▅▅▅▄
clip_fraction,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▂▁▁▂▅▁▁▁▁▁▁▁▁▁▁▁
ema_norm_reward,█▇▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
entropy_loss,▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▄▄▃▃▄▇████▆▆▆█▇▇▇▅▅▅▅▅▅▅▄
episode_length_mean,▃▄▂▁▃▅▄▂▆▂▃█▂▄▇▃▂█▂▃▄▂▃▂▂▃▂▃▄▂▃▄▄▄▂▂▁▂▂▂
episode_reward_max,▁▃▁▆▆▁▁▃▃▁▁▃▃▆▃▁█▆▁▃▁▁▁▃▁▁▃▆▆▆▁▃▆▃▃▁▁▃▃▃
episode_reward_mean,▁▁▅▁██▁▁▅▅▅█▅▅▁▁█▅█▁▁▅▅▁▅█▅▁▅▅▁▁▅▅▁▁▁▅▁█
episode_reward_min,▁▁▃▁▁▆▆▁▃▁▃▃▁▃▁▃▃▁▆▆▁▃▁▁▃▁▁▁▁▁▁▁▃▃▃█▁▁▁▃
episodes_completed,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇█████
explained_variance,▁▁▁▁▅▆▇▇▇▇▇██▇▇█████████████████████████
+1,...


-19.68

In [ ]:
[0.7930864926220205,0.7229910906298003,0.7578080645161289]